# Medical RAG on MedQA-USMLE: Stage 2 (dense retrieval + LLM)
Run on Colab with a **GPU runtime** (Runtime → Change runtime type → T4 GPU).

**Before you start:** upload `medrag_repo.zip` and your `data_clean` folder (zipped) to Google Drive under `MyDrive/medrag/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/work && cd /content/work && unzip -q -o /content/drive/MyDrive/medrag/medrag_repo.zip && unzip -q -o /content/drive/MyDrive/medrag/data_clean.zip
%cd /content/work/medrag
!pip -q install faiss-cpu rank_bm25 sentence-transformers openai scikit-learn accelerate bitsandbytes

## 1. Rebuild the chunks (same settings as Stage 1)

In [ ]:
import os
os.environ['MEDQA_DIR'] = '/content/work/data_clean'
!python prep.py

## 2. LLM access
**Option A (default, free, no key):** run an open-weights model on the Colab GPU. Greedy decoding makes it deterministic and fully reproducible, which reviewers like.
`Qwen/Qwen2.5-3B-Instruct` fits a T4 in fp16. For a stronger reader use `Qwen/Qwen2.5-7B-Instruct` with `--load-4bit`, which is slower.

**Option B:** any OpenAI-compatible API. Put the key in Colab **Secrets** (key icon) as `LLM_API_KEY`.

| Provider | `LLM_BASE_URL` | example `LLM_MODEL` |
|---|---|---|
| OpenAI | https://api.openai.com/v1 | gpt-4o-mini |
| Groq | https://api.groq.com/openai/v1 | llama-3.1-8b-instant |
| OpenRouter | https://openrouter.ai/api/v1 | meta-llama/llama-3.1-8b-instruct |

Pick **one model and keep it fixed** for the whole study.

In [ ]:
BACKEND = 'hf'                        # 'hf' (local GPU, no key)  or  'openai' (API)
LLM_MODEL = 'Qwen/Qwen2.5-3B-Instruct'  # API example: 'gpt-4o-mini'
LLM_BASE_URL = 'https://api.openai.com/v1'  # only used when BACKEND == 'openai'
EXTRA = ''                            # e.g. '--load-4bit' for Qwen/Qwen2.5-7B-Instruct
if BACKEND == 'openai':
    from google.colab import userdata
    os.environ['LLM_API_KEY'] = userdata.get('LLM_API_KEY')
RUNS = '/content/drive/MyDrive/medrag/runs'   # results go to Drive, so a disconnect loses nothing
COMMON = f"--data $MEDQA_DIR --chunks data/chunks_c512.jsonl --backend {BACKEND} --llm-model {LLM_MODEL} --llm-base-url {LLM_BASE_URL} {EXTRA} --out {RUNS}"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Headroom check (100 dev questions, closed-book)
If closed-book accuracy is already above ~85%, RAG has little room to help, so consider a smaller model.
With Qwen2.5-3B, expect roughly 45–60% closed-book. That leaves room for retrieval to help or hurt.

In [ ]:
!python stage2_llm.py $COMMON --split dev500 --limit 100 --configs closed

## 4. Dev sweeps (500 fixed dev questions)
Runs are cached and resumable. If Colab disconnects, re-run the cell and it continues where it stopped.

Time estimate on a T4 with the 3B model: about 1–2 s per call, so roughly 2–3 hours for this cell (map-rerank makes 5 calls per question). If a session ends, reconnect and re-run from section 0; finished rows are skipped.

In [ ]:
!python stage2_llm.py $COMMON --split dev500 --configs closed,random5,bm25_k1,bm25_k3,bm25_k5,bm25_k10,bm25_rerank20_k5,bm25_k10_compress0.7,bm25_k10_compress0.8,bm25_k10_compress0.9,bm25_maprerank_k5,bm25_reciprocal_k5

In [ ]:
# dense retrieval, one embedding model per run (the corpus embedding is cached on Drive)
for emb in ['BAAI/bge-small-en-v1.5', 'BAAI/bge-base-en-v1.5', 'medcpt']:
    tag = emb.split('/')[-1]
    !python stage2_llm.py $COMMON --split dev500 --embed {emb} --out /content/drive/MyDrive/medrag/runs_{tag} --configs dense_k1,dense_k3,dense_k5,dense_k10,dense_q_k5,dense_rerank20_k5

In [ ]:
!python analyze_stage2.py /content/drive/MyDrive/medrag/runs/dev500

## 5. Final test run (only the configurations chosen on dev)
Edit the list after looking at the **dev** results. Run the test set once and don't tune afterwards.

In [ ]:
FINAL = 'closed,random5,bm25_k5,bm25_k10,bm25_rerank20_k5,bm25_k10_compress0.8,bm25_maprerank_k5,bm25_reciprocal_k5'
!python stage2_llm.py $COMMON --split test --configs {FINAL}
!python analyze_stage2.py /content/drive/MyDrive/medrag/runs/test